# Sensor Comparison

This notebook compares the imaging characteristics of HyPlan's built-in airborne sensors.
We examine how field of view, ground sample distance, swath width, and critical ground speed
vary across sensors and flight altitudes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from hyplan.sensors import SENSOR_REGISTRY, create_sensor, LineScanner
from hyplan.units import ureg

## 1. Sensor Registry Overview

List all built-in line-scanner sensors with their key specifications.

In [ ]:
rows = []
for name, cls in sorted(SENSOR_REGISTRY.items()):
    sensor = cls()
    if isinstance(sensor, LineScanner):
        rows.append({
            "Sensor": sensor.name,
            "FOV (deg)": sensor.fov,
            "Pixels": sensor.across_track_pixels,
            "Frame Rate (Hz)": sensor.frame_rate.to(ureg.Hz).magnitude,
            "IFOV (mrad)": round(sensor.ifov * 1000 * np.pi / 180, 3),
        })

spec_df = pd.DataFrame(rows)
spec_df

## 2. Ground Sample Distance vs. Altitude

Compare nadir GSD for each sensor across a range of flight altitudes AGL.

In [ ]:
altitudes_m = np.arange(1000, 20001, 1000)
sensors_to_compare = ["AVIRIS-3", "AVIRIS-4", "AVIRIS-NG", "HyTES", "PRISM", "MASTER"]

fig, ax = plt.subplots(figsize=(10, 6))

for sensor_name in sensors_to_compare:
    sensor = create_sensor(sensor_name)
    gsd_vals = [
        sensor.ground_sample_distance(ureg.Quantity(alt, "meter")).to(ureg.meter).magnitude
        for alt in altitudes_m
    ]
    ax.plot(altitudes_m / 1000, gsd_vals, marker="o", markersize=4, label=sensor.name)

ax.set_xlabel("Altitude AGL (km)")
ax.set_ylabel("Nadir GSD (m)")
ax.set_title("Ground Sample Distance vs. Flight Altitude")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Swath Width vs. Altitude

Wider swaths mean fewer flight lines to cover an area, but at the cost of coarser GSD.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for sensor_name in sensors_to_compare:
    sensor = create_sensor(sensor_name)
    swath_vals = [
        sensor.swath_width(ureg.Quantity(alt, "meter")).to(ureg.km).magnitude
        for alt in altitudes_m
    ]
    ax.plot(altitudes_m / 1000, swath_vals, marker="s", markersize=4, label=sensor.name)

ax.set_xlabel("Altitude AGL (km)")
ax.set_ylabel("Swath Width (km)")
ax.set_title("Swath Width vs. Flight Altitude")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Critical Ground Speed

The critical ground speed is the maximum aircraft speed that maintains contiguous
along-track sampling (no gaps between scan lines). Exceeding this speed causes
under-sampling.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for sensor_name in sensors_to_compare:
    sensor = create_sensor(sensor_name)
    speed_vals = [
        sensor.critical_ground_speed(ureg.Quantity(alt, "meter")).to(ureg.knot).magnitude
        for alt in altitudes_m
    ]
    ax.plot(altitudes_m / 1000, speed_vals, marker="^", markersize=4, label=sensor.name)

ax.set_xlabel("Altitude AGL (km)")
ax.set_ylabel("Critical Ground Speed (knots)")
ax.set_title("Maximum Speed for Contiguous Along-Track Sampling")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Altitude Required for a Target GSD

Given a desired ground sample distance, what altitude must the aircraft fly at?

In [ ]:
target_gsds_m = [1, 2, 3, 5, 8, 10, 15, 20, 30]

rows = []
for sensor_name in sensors_to_compare:
    sensor = create_sensor(sensor_name)
    for gsd in target_gsds_m:
        alt = sensor.altitude_agl_for_ground_sample_distance(
            ureg.Quantity(gsd, "meter")
        ).to(ureg.meter).magnitude
        rows.append({
            "Sensor": sensor.name,
            "Target GSD (m)": gsd,
            "Required Altitude AGL (m)": round(alt),
        })

alt_df = pd.DataFrame(rows)
pivot = alt_df.pivot(index="Target GSD (m)", columns="Sensor", values="Required Altitude AGL (m)")
pivot

## 6. Along-Track Pixel Size at Typical Airspeeds

At a given aircraft speed, the along-track pixel size depends on the sensor's frame rate.
Compare along-track resolution at 150 knots for different sensors and altitudes.

In [ ]:
speed = ureg.Quantity(150, "knot")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sensor_name in sensors_to_compare:
    sensor = create_sensor(sensor_name)
    at_sizes = [
        sensor.along_track_pixel_size(speed).to(ureg.meter).magnitude
        for _ in altitudes_m
    ]
    ct_sizes = [
        sensor.ground_sample_distance(ureg.Quantity(alt, "meter")).to(ureg.meter).magnitude
        for alt in altitudes_m
    ]
    axes[0].plot(altitudes_m / 1000, at_sizes, marker="o", markersize=3, label=sensor.name)
    axes[1].plot(altitudes_m / 1000, ct_sizes, marker="o", markersize=3, label=sensor.name)

axes[0].set_xlabel("Altitude AGL (km)")
axes[0].set_ylabel("Along-Track Pixel Size (m)")
axes[0].set_title(f"Along-Track Pixel Size at {speed:~}")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Altitude AGL (km)")
axes[1].set_ylabel("Cross-Track GSD (m)")
axes[1].set_title("Cross-Track GSD (Nadir)")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()